# ব্যয় দাবি বিশ্লেষণ

এই নোটবুকটি প্রদর্শন করে কীভাবে প্লাগইন ব্যবহার করে এজেন্ট তৈরি করা যায় যা স্থানীয় রসিদ চিত্র থেকে ভ্রমণ ব্যয় প্রসেস করে, একটি ব্যয় দাবি ইমেল তৈরি করে এবং পাই চাট দিয়ে ব্যয় ডেটা দৃশ্যায়ন করে। এজেন্টরা কাজের প্রসঙ্গ অনুসারে গতিশীলভাবে ফাংশন নির্বাচন করে।

ধাপসমূহ:
1. OCR এজেন্ট স্থানীয় রসিদ চিত্রটি প্রসেস করে এবং ভ্রমণ ব্যয় ডেটা আহরণ করে।
2. ইমেল এজেন্ট একটি ব্যয় দাবি ইমেল তৈরি করে।

### ভ্রমণ ব্যয় পরিস্থিতির একটি উদাহরণ:
ভাবুন আপনি একজন কর্মচারী হিসেবে একটি ব্যবসায়িক বৈঠকের জন্য অন্য শহরে ভ্রমণ করছেন। আপনার কোম্পানির একটি নীতি আছে যা সমস্ত যুক্তিসঙ্গত ভ্রমণ সম্পর্কিত ব্যয় ফেরত দেয়। এখানে সম্ভাব্য ভ্রমণ ব্যয়ের একটি বিভাজন:
- পরিবহন:
আপনার বাড়ি শহর থেকে গন্তব্য শহরে রাউন্ড ট্রিপের জন্য বিমানের ভাড়া।
বিমানবন্দর থেকে এবং বিমানবন্দরে যাওয়ার জন্য ট্যাক্সি বা রাইড-হেইলিং সার্ভিস।
গন্তব্য শহরে স্থানীয় পরিবহন (যেমন পাবলিক ট্রানজিট, বিনিয়োগ করা গাড়ি, বা ট্যাক্সি)।

- আবাসন:
মিটিং ভেন্যুর কাছাকাছি একটি মাঝারি মানের ব্যবসায়িক হোটেলে তিন রাত থাকার ব্যবস্থা।

- খাবার:
প্রতিদিনের খাবারের ভাতা (নাস্তা, মধ্যাহ্নভোজন এবং রাতের খাবার) কোম্পানির পার ডায়েম নীতির উপর ভিত্তি করে।

- বিবিধ ব্যয়:
বিমানবন্দরে পার্কিং ফি।
হোটেলে ইন্টারনেট অ্যাক্সেস চার্জ।
টিপস বা ছোটো সার্ভিস চার্জ।

- নথি প্রস্তুতি:
আপনি সমস্ত রসিদ (উড়োজাহাজ, ট্যাক্সি, হোটেল, খাবার, ইত্যাদি) এবং একটি সম্পূর্ণ ব্যয় রিপোর্ট জমা দেন ফেরতের জন্য।


## প্রয়োজনীয় লাইব্রেরি ইম্পোর্ট করুন

নোটবুকের জন্য প্রয়োজনীয় লাইব্রেরি এবং মডিউলগুলি ইম্পোর্ট করুন।


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## ব্যয় মডেল সংজ্ঞায়িত করুন

একটি পৃথক খরচের জন্য একটি Pydantic মডেল তৈরি করুন এবং একটি ExpenseFormatter ক্লাস তৈরি করুন যা ব্যবহারকারীর কোয়েরিকে গঠনমূলক ব্যয় তথ্যতে রূপান্তর করবে।

প্রতিটি ব্যয় নিম্নলিখিত ফরম্যাটে উপস্থাপিত হবে:
`{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - ইমেইল তৈরি করা

একটি টুল ফাংশন তৈরি করুন যা একটি খরচ দাবি জমা দেওয়ার জন্য ইমেইল তৈরি করে।
- এই টুলটি Microsoft Agent Framework থেকে `@tool` ডেকোরেটর ব্যবহার করে।
- এটি খরচের মোট পরিমাণ হিসাব করে এবং বিস্তারিত তথ্য ইমেইল বডিতে ফরম্যাট করে।


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# ভ্রমণ খরচ রশিদ ইমেজ থেকে বের করার জন্য টুল

রশিদ ইমেজ থেকে ভ্রমণ খরচ বের করার জন্য একটি টুল ফাংশন তৈরি করুন।
- এই টুলটি Microsoft Agent Framework এর `@tool` ডেকোরেটর ব্যবহার করে।
- এটি রশিদ ইমেজ পড়ে, সেটিকে base64 হিসেবে এনকোড করে, এবং এজেন্টের বিশ্লেষণের জন্য ডাটা URI প্রদান করে।


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Processing Expenses

পরিবহন করণীয় এজেন্টগুলি সংজ্ঞায়িত করুন এবং `WorkflowBuilder` ব্যবহার করে একটি ধারাবাহিক কার্যপ্রবাহের সাথে সংযুক্ত করুন।
- OCR এজেন্টটি `load_receipt_image` টুল ব্যবহার করে রসিদ ছবির থেকে গঠনমূলক খরচের তথ্য বের করে।
- Email এজেন্টটি প্রাপ্ত তথ্য নিয়ে `generate_expense_email` টুল ব্যবহার করে একটি পেশাদারী খরচ দাবি ইমেইল তৈরি করে।
- `WorkflowBuilder` `add_edge` সহ একটি ধারাবাহিক পাইপলাইন তৈরি করে: OCR এজেন্ট → Email এজেন্ট।


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

সিকুয়েন্সিয়াল ওয়ার্কফ্লো তৈরি করুন এবং চালান যাতে রসিদের ছবি প্রক্রিয়া করে এবং ব্যয় দাবি ইমেল তৈরি করা যায়।


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকারোক্তি**:  
এই নথিটি কৃত্রিম বুদ্ধিমত্তা অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসাধ্য সঠিকতার জন্য চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ভুল বা ত্রুটি থাকতে পারে বলে অনুগ্রহ করে দয়া করে সতর্ক থাকুন। অরিজিনাল নথিটি তার স্বাভাবিক ভাষায় প্রাসঙ্গিক এবং সর্বাধিক বিশ্বস্ত উৎস হিসেবে বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ পরামর্শ দেওয়া হয়। এই অনুবাদের ব্যবহারে উদ্ভূত কোনো বিভ্রান্তি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
